## 文本转语音(Text-to-audio)
### 英文语音合成
 **任务**: Text-to-Audio (TTS)

 **模型**: microsoft/speecht5_tts（~586MB） + microsoft/speecht5_hifigan（~50MB）

 **架构**: SpeechT5 Transformer 编码器-解码器 + HiFi-GAN 神经声码器

 **数据集**：Matthijs/cmu-arctic-xvectors (~18MB)

 **框架**: MindNLP 0.4.1 + MindSpore 2.6.0

 **硬件**: OrangePi Aipro 20T24G

     运行本 notebook 文件 前，需手动修改3个库文件共 4 处（具体见README）），此修改不可跳过，否则代码执行会遇到报错。
 （注：提供的 .ipynb文件直接预览时，可能无法渲染出如README演示中的按键，需要依次执行各个cell后才能正常显示，具体效果见README.md中Demo示例图）

In [1]:
!python -V
!pip show mindnlp

Python 3.9.2
Name: mindnlp
Version: 0.4.1
Summary: An open source natural language processing research tool box. Git version: [sha1]:22221f40, [branch]: (HEAD -> master, tag: v0.4.1, origin/master)
Home-page: https://github.com/mindlab-ai/mindnlp/tree/master/
Author: MindSpore Team
Author-email: 
License: Apache 2.0
Location: /home/HwHiAiUser/.local/lib/python3.9/site-packages
Requires: addict, datasets, evaluate, mindspore, ml-dtypes, pillow, pyctcdecode, pytest, regex, requests, safetensors, sentencepiece, tokenizers, tqdm
Required-by: 


In [2]:
%pip install scipy soundfile ipywidgets

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
import mindspore as ms

print(f"MindSpore 版本: {ms.__version__}")
print(f"当前运行设备: {ms.get_context('device_target')}")

/usr/local/miniconda3/lib/python3.9/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/miniconda3/lib/python3.9/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/miniconda3/lib/python3.9/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/miniconda3/lib/python3.9/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


MindSpore 版本: 2.6.0
当前运行设备: Ascend


### 配置

In [4]:
# 日志配置
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)

In [5]:
# 模型输出目录配置
import os
OUTPUT_DIR = "./tta_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_ID = "microsoft/speecht5_tts"
VOCODER_ID = "microsoft/speecht5_hifigan"

logger.info("音频输出目录: %s", os.path.abspath(OUTPUT_DIR))
logger.info("TTS 模型: %s", MODEL_ID)
logger.info("声码器模型: %s", VOCODER_ID)

2026-04-27 00:20:42,362 [INFO] 音频输出目录: /home/HwHiAiUser/TTA/tta_outputs
2026-04-27 00:20:42,368 [INFO] TTS 模型: microsoft/speecht5_tts
2026-04-27 00:20:42,373 [INFO] 声码器模型: microsoft/speecht5_hifigan


### 模型下载与加载

In [6]:
import time
import mindnlp
from mindnlp.transformers import SpeechT5ForTextToSpeech, SpeechT5Processor, SpeechT5HifiGan

logger.info("正在加载模型： %s（首次运行需等待下载 ～590MB）", MODEL_ID)
start_time = time.time()
processor = SpeechT5Processor.from_pretrained(MODEL_ID)
model = SpeechT5ForTextToSpeech.from_pretrained(MODEL_ID)
elapsed = time.time() - start_time
logger.info("TTS模型加载完成。 耗时： %.1fs", elapsed)


[WARNING] ME(6095:255086501703712,MainProcess):2026-04-27-00:20:45.569.406 [mindspore/context.py:1402] For 'context.set_context', the parameter 'ascend_config' will be deprecated and removed in a future version. Please use the api mindspore.device_context.ascend.op_precision.precision_mode(),
                                                       mindspore.device_context.ascend.op_precision.op_precision_mode(),
                                                       mindspore.device_context.ascend.op_precision.matmul_allow_hf32(),
                                                       mindspore.device_context.ascend.op_precision.conv_allow_hf32(),
                                                       mindspore.device_context.ascend.op_tuning.op_compile() instead.
Building prefix dict from the default dictionary ...
2026-04-27 00:20:48,733 [DEBUG] Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
2026-04-27 00:20:48,739 [DEBUG] Loading model 

In [11]:
logger.info("正在加载声码器： %s（首次运行需等待下载 ～50MB）", VOCODER_ID)
start_time = time.time()
vocoder = SpeechT5HifiGan.from_pretrained(VOCODER_ID)
elapsed = time.time() - start_time
logger.info("声码器加载完成。 耗时： %.1fs", elapsed)

2026-04-27 00:23:50,677 [INFO] 正在加载声码器： microsoft/speecht5_hifigan（首次运行需等待下载 ～50MB）
2026-04-27 00:24:15,531 [INFO] 声码器加载完成。 耗时： 24.8s


In [12]:
# 设置推理模式
model.set_train(False)
vocoder.set_train(False)

SpeechT5HifiGan(
  (conv_pre): Conv1d(80, 512, kernel_size=(7,), stride=(1,), padding=(3,))
  (upsampler): ModuleList(
    (0): ConvTranspose1d(512, 256, kernel_size=(8,), stride=(4,), padding=(2,))
    (1): ConvTranspose1d(256, 128, kernel_size=(8,), stride=(4,), padding=(2,))
    (2): ConvTranspose1d(128, 64, kernel_size=(8,), stride=(4,), padding=(2,))
    (3): ConvTranspose1d(64, 32, kernel_size=(8,), stride=(4,), padding=(2,))
  )
  (resblocks): ModuleList(
    (0): HifiGanResidualBlock(
      (convs1): ModuleList(
        (0): Conv1d(256, 256, kernel_size=(3,), stride=(1,), padding=(1,))
        (1): Conv1d(256, 256, kernel_size=(3,), stride=(1,), padding=(3,), dilation=(3,))
        (2): Conv1d(256, 256, kernel_size=(3,), stride=(1,), padding=(5,), dilation=(5,))
      )
      (convs2): ModuleList(
        (0-2): 3 x Conv1d(256, 256, kernel_size=(3,), stride=(1,), padding=(1,))
      )
    )
    (1): HifiGanResidualBlock(
      (convs1): ModuleList(
        (0): Conv1d(256, 256,

In [13]:
# 设置 NPU 推理：使用 fp16，避免数据类型不匹配
model = model.half()

# SpeechT5 postnet 含 BatchNorm（昇腾 310B 不支持），此处以恒等映射绕过
model.speech_decoder_postnet.postnet = lambda x: x
logger.info("模型已配置为 NPU 推理（fp16，postnet 已适配）")

2026-04-27 00:24:34,097 [INFO] 模型已配置为 NPU 推理（fp16，postnet 已适配）


### 说话人嵌入配置（首次运行自动下载获取数据集）

In [14]:
import numpy as np
from datasets import load_dataset

SAMPLE_RATE = 16000  # SpeechT5 固定采样率

logger.info("正在加载说话人嵌入数据集（首次运行需下载 ～18MB）")
_xvec_ds  = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
XVEC_MAX  = len(_xvec_ds) - 1          # 可选范围 0 ～ XVEC_MAX
SPEAKER_IDX = 7306                     # 默认说话人（可选范围：0 ～ XVEC_MAX）
SPEAKER_EMB = ms.Tensor(
    np.array(_xvec_ds[SPEAKER_IDX]["xvector"], dtype=np.float32)[None],
    dtype=ms.float16,
)
logger.info("采样率: %d Hz | 参数量: %.1fM", SAMPLE_RATE, sum(p.numel() for p in model.get_parameters()) / 1e6)
logger.info("说话人嵌入已加载（index=%d，可选范围：0～%d）", SPEAKER_IDX, XVEC_MAX)

2026-04-27 00:24:42,054 [INFO] 正在加载说话人嵌入数据集（首次运行需下载 ～18MB）
2026-04-27 00:25:00,873 [INFO] 采样率: 16000 Hz | 参数量: 144.4M
2026-04-27 00:25:00,895 [INFO] 说话人嵌入已加载（index=7306，可选范围：0～7930）


### 定义核心推理函数

In [15]:
import numpy as np
import soundfile as sf

_SR = SAMPLE_RATE  # BoardAudioPlayer 默认参数引用


def _hifigan(mel_np: np.ndarray) -> np.ndarray:
    """HiFi-GAN 神经声码器"""
    mel_t = ms.Tensor(mel_np[None], dtype=ms.float32)
    wav = vocoder(mel_t)
    return wav.squeeze().asnumpy().astype(np.float32)


def tta_infer(text: str, speaker_embeddings=None, threshold: float = 0.5, 
              save_path: str = None, verbose: bool = True,) -> tuple:
    """SpeechT5 文本转语音推理（HiFi-GAN 声码器）"""
    if speaker_embeddings is None:
        speaker_embeddings = SPEAKER_EMB

    if verbose:
        logger.info("输入文本: %s", text)

    inputs = processor(text=text, return_tensors="ms")

    t_start = time.time()
    spectrogram = model.generate_speech(
        inputs["input_ids"],
        speaker_embeddings,
        threshold=threshold,
    )
    infer_time = time.time() - t_start

    mel_np = spectrogram.asnumpy().astype(np.float32)
    audio = _hifigan(mel_np)

    audio_dur = len(audio) / _SR
    rtf = infer_time / audio_dur

    if verbose:
        logger.info("推理耗时: %.2fs | 音频时长: %.2fs | RTF: %.3f", infer_time, audio_dur, rtf)

    if save_path:
        sf.write(save_path, audio, _SR)
        if verbose:
            logger.info("已保存: %s", save_path)

    return audio, _SR

logger.info("推理函数定义完成")

2026-04-27 00:25:04,789 [INFO] 推理函数定义完成


## Orangepi Aipro 20T 板载音频播放适配

### 问题：`IPython.display.Audio` 在本机浏览器无声

`IPython.display.Audio` 生成的是 HTML5 `<audio>` 播放器，由**浏览器**调用系统音频栈（PulseAudio → ALSA）来出声。

但 OrangePi AIPro 20T 的音频驱动走的是华为昇腾 **MPI（媒体处理接口）** 框架，内核未编译标准 SoC 音频驱动（`CONFIG_SND_SOC` 未启用），ALSA 找不到任何声卡：

命令输出：
```
$ aplay -l
aplay: device_list:274: no soundcards found...
```

因此：
- 在开发板**本地浏览器**打开 Jupyter → 浏览器拿不到音频设备 → **无声**
- 从**远程电脑**访问 Jupyter → 浏览器用远程机器的声卡 → **有声**

### 解决方案：`play_on_board()`

下方 cell 定义了 `play_on_board()`，绕过 ALSA，直接调用板载 MPI 音频接口：

1. 将 numpy 波形（或 WAV 文件）重采样至 **48kHz**（`sample_audio` 的要求）
2. 转为 **16bit 单声道 PCM** 并写入临时文件
3. 调用 `/opt/opi_test/audio/sample_audio play 2 <pcm>` 通过 MPI 输出到开发板的圆孔**耳机接口**

此后所有推理 cell 均同时调用：
* `play_audio()`（保留 HTML5 播放器）->对应黑色按键 
* `play_on_board()`（板载实际出声）->对应绿色按键。

In [16]:
import os
import subprocess
import tempfile
from math import gcd
from scipy.signal import resample_poly

# 板载 MPI 音频常量
_BOARD_SR = 48000
_BOARD_DEV = 2
_MPI_LIBS = (
    "/usr/local/Ascend/ascend-toolkit/latest/lib64:"
    "/var/davinci/driver/lib64:"
    "/usr/local/Ascend/driver/lib64"
)
_SAMPLE_AUDIO = "/opt/opi_test/audio/sample_audio"
_SAMPLE_AUDIO_CWD = "/opt/opi_test/audio"

import threading, time
import ipywidgets as widgets
from IPython.display import display as _ipy_display, Audio as _IpyAudio


class BoardAudioPlayer:
    """可重复点击的板载音频播放 Widget。

    show() 同时展示：
      · 板载 ▶/■ 按钮 + 进度条（调用 MPI 实际出声）
      · HTML5 <audio> 播放器（远程浏览器访问时有声）
    """

    def __init__(self, audio, sample_rate: int = _SR, title: str = "", device: int = _BOARD_DEV):
        self._title = title
        self._device = device
        self._proc = None
        self._lock = threading.Lock()
        self._raw = (audio if not isinstance(audio, (str, os.PathLike)) else None, sample_rate)

        # 转换为 48kHz 16bit PCM，存入临时文件（只转换一次，可多次播放）
        self._tmp, self._duration = self._to_pcm(audio, sample_rate)

        # ===== Widget 组件 =====
        self._btn = widgets.Button(
            description="▶  播放",
            button_style="success",
            layout=widgets.Layout(width="110px", height="36px"),
        )
        self._bar = widgets.FloatProgress(
            value=0, min=0, max=max(self._duration, 0.001),
            bar_style="info",
            layout=widgets.Layout(width="260px", height="18px"),
        )
        self._lbl = widgets.Label(
            value=f"00:00 / {self._fmt(self._duration)}",
            layout=widgets.Layout(width="120px"),
        )
        self._btn.on_click(self._toggle)

    # ===== 工具 =====
    @staticmethod
    def _fmt(s: float) -> str:
        m, s = divmod(int(s), 60)
        return f"{m:02d}:{s:02d}"

    def _to_pcm(self, audio, sr):
        if isinstance(audio, (str, os.PathLike)):
            import soundfile as _sf
            audio, sr = _sf.read(str(audio), dtype="float32")
            self._raw = (audio, sr)
        if sr != _BOARD_SR:
            g = gcd(sr, _BOARD_SR)
            audio = resample_poly(audio, _BOARD_SR // g, sr // g)
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        pcm = (audio * 32767).clip(-32768, 32767).astype(np.int16)
        f = tempfile.NamedTemporaryFile(suffix=".pcm", delete=False)
        pcm.tofile(f); f.close()
        return f.name, len(pcm) / _BOARD_SR

    # ===== 播放控制 =====
    def _toggle(self, _=None):
        with self._lock:
            if self._proc and self._proc.poll() is None:
                self._do_stop()
            else:
                self._do_play()

    def _do_play(self):
        env = os.environ.copy()
        env["LD_LIBRARY_PATH"] = _MPI_LIBS
        self._proc = subprocess.Popen(
            [_SAMPLE_AUDIO, "play", str(self._device), self._tmp],
            cwd=_SAMPLE_AUDIO_CWD, env=env,
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        self._btn.description  = "■  结束"
        self._btn.button_style = "danger"
        threading.Thread(target=self._monitor, daemon=True).start()

    def _do_stop(self):
        if self._proc and self._proc.poll() is None:
            self._proc.terminate()
        self._proc = None
        self._btn.description  = "▶  播放"
        self._btn.button_style = "success"
        self._bar.value = 0
        self._lbl.value = f"00:00 / {self._fmt(self._duration)}"

    def _monitor(self):
        t0 = time.time()
        while self._proc and self._proc.poll() is None:
            elapsed = time.time() - t0
            self._bar.value = min(elapsed, self._duration)
            self._lbl.value = f"{self._fmt(elapsed)} / {self._fmt(self._duration)}"
            time.sleep(0.15)
        with self._lock:
            if self._proc is not None:      # 自然播放结束（非手动停止）
                self._do_stop()

    def stop(self):
        """外部调用：强制停止当前播放。"""
        with self._lock:
            self._do_stop()

    # ===== 展示 Widget =====
    def show(self):
        header = (widgets.HTML(f"<span style='font-size:13px'><b>🔊 {self._title}</b></span>")
                  if self._title else widgets.HTML(""))
        row = widgets.HBox(
            [self._btn, self._bar, self._lbl],
            layout=widgets.Layout(align_items="center", gap="8px"),
        )
        board_ui = widgets.VBox([header, row])

        # 板载播放器 Widget
        _ipy_display(board_ui)
        # HTML5 播放器（远程浏览器访问时有声）
        wav, sr = self._raw
        if wav is not None:
            _ipy_display(_IpyAudio(wav, rate=sr))
        return self

    def __del__(self):
        try:
            if os.path.exists(self._tmp):
                os.unlink(self._tmp)
        except Exception:
            pass

logger.info("BoardAudioPlayer 已定义")

2026-04-27 00:25:12,573 [INFO] BoardAudioPlayer 已定义


### 基础推理示例

In [17]:
logger.info("--- 基础推理示例：说话人嵌入对比 ---")
text_basic = "Hello, welcome to the text-to-speech basic demo running on Ascend N P U."

# 对比 1：零向量嵌入（基线）
logger.info("对比 1/2：零向量嵌入（基线）")
_zero_emb = ms.Tensor(np.zeros((1, 512), dtype=np.float16))
wav_zero, sr = tta_infer(
    text_basic,
    speaker_embeddings=_zero_emb,
    save_path=os.path.join(OUTPUT_DIR, "basic_zero.wav"),
)
BoardAudioPlayer(wav_zero, sr, title="零向量嵌入（基线）").show()

# 对比 2：数据集嵌入
logger.info("对比 2/2：数据集嵌入（index=%d）", SPEAKER_IDX)
wav_xvec, sr = tta_infer(
    text_basic,
    speaker_embeddings=SPEAKER_EMB,
    save_path=os.path.join(OUTPUT_DIR, "basic_xvec.wav"),
)
BoardAudioPlayer(wav_xvec, sr, title=f"数据集嵌入（index={SPEAKER_IDX}）").show()

logger.info("由上对比，更推荐使用数据集向量")

2026-04-27 00:25:22,149 [INFO] --- 基础推理示例：说话人嵌入对比 ---
2026-04-27 00:25:22,157 [INFO] 对比 1/2：零向量嵌入（基线）
2026-04-27 00:25:22,161 [INFO] 输入文本: Hello, welcome to the text-to-speech basic demo running on Ascend N P U.


.

2026-04-27 00:26:41,189 [INFO] 推理耗时: 67.16s | 音频时长: 3.23s | RTF: 20.779
2026-04-27 00:26:41,223 [INFO] 已保存: ./tta_outputs/basic_zero.wav


2026-04-27 00:26:41,379 [INFO] 对比 2/2：数据集嵌入（index=7306）
2026-04-27 00:26:41,415 [INFO] 输入文本: Hello, welcome to the text-to-speech basic demo running on Ascend N P U.
2026-04-27 00:27:39,890 [INFO] 推理耗时: 57.11s | 音频时长: 5.06s | RTF: 11.296
2026-04-27 00:27:39,976 [INFO] 已保存: ./tta_outputs/basic_xvec.wav


2026-04-27 00:27:40,073 [INFO] 由上对比，更推荐使用数据集向量


### 交互 Demo（Ipywidgets）

In [18]:
import os
from IPython.display import display

try:
    import ipywidgets as widgets
    from IPython.display import clear_output

    display(widgets.HTML("""
    <style>
    .widget-slider .noUi-handle {
        background: #ff7043 !important; border-color: #e64a19 !important;
        box-shadow: 0 0 0 3px #ffccbc !important;
    }
    .widget-slider .noUi-connect { background: #ff7043 !important; }
    .tta-section {
        font-weight: bold; font-size: 13px; color: #444;
        margin: 12px 0 4px; border-left: 3px solid #ff7043; padding-left: 6px;
    }
    </style>"""))

    def _section(title):
        return widgets.HTML(f"<div class='tta-section'>{title}</div>")

    #  文本输入
    text_input = widgets.Textarea(
        value=(
        "Hello! I am currently running on the OrangePi AIpro 20T, "
        "a powerful edge computing device for artificial intelligence. "
        "Today, we are testing the Speech TTS model. The weather outside is sunny, "
        "and the room temperature is optimal for computing. How does my voice sound? "
        "Let's explore the future of on-device AI together."
        ),
        description="",
        layout=widgets.Layout(width="99%", height="72px"),
    )

    # Threshold 
    thr_slider = widgets.FloatSlider(
        value=0.5, min=0.1, max=0.9, step=0.01,
        description="threshold:", readout_format=".2f",
        style={"description_width": "90px"},
        layout=widgets.Layout(width="55%"),
    )
    thr_desc = widgets.HTML(
        "<span style='color:#888;font-size:11px;margin-left:8px'>"
        "值越高语音越完整，值越低越早截断</span>"
    )

    # 说话人嵌入：零向量 
    chk_zero = widgets.Checkbox(
        value=True, description="零向量（基线）",
        indent=False, layout=widgets.Layout(width="155px"),
    )

    # 说话人嵌入：数据集（每个 slot 独立输入框）
    chk_xvec = widgets.Checkbox(
        value=True, description="数据集嵌入",
        indent=False, layout=widgets.Layout(width="115px"),
    )
    xvec_count = widgets.BoundedIntText(
        value=1, min=1, max=5,
        description="数量 (max 5):",
        style={"description_width": "90px"},
        layout=widgets.Layout(width="160px"),
    )

    _DEFAULT_INDICES = [SPEAKER_IDX, 1000, 3000, 5000, 7000]
    xvec_inputs = [
        widgets.BoundedIntText(
            value=_DEFAULT_INDICES[i], min=0, max=XVEC_MAX,
            description=f"index {i+1}:",
            style={"description_width": "60px"},
            layout=widgets.Layout(width="180px"),
        )
        for i in range(5)
    ]
    xvec_hint = widgets.HTML(
        f"<span style='color:#888;font-size:11px'>可选范围：0 ～ {XVEC_MAX}</span>"
    )
    xvec_inputs_box = widgets.VBox([
        widgets.HBox(xvec_inputs[:3]),
        widgets.HBox(xvec_inputs[3:]),
        xvec_hint,
    ])

    def _sync_xvec_vis(change=None):
        show = chk_xvec.value
        xvec_count.layout.visibility = "visible" if show else "hidden"
        xvec_inputs_box.layout.display = "" if show else "none"
        if show:
            _sync_count()

    def _sync_count(change=None):
        n = xvec_count.value
        for i, inp in enumerate(xvec_inputs):
            inp.layout.display = "" if i < n else "none"

    chk_xvec.observe(_sync_xvec_vis, names="value")
    xvec_count.observe(_sync_count, names="value")
    _sync_xvec_vis()  # 初始化

    #  按钮 & 输出 
    run_btn = widgets.Button(
        description="开始合成",
        button_style="primary",
        layout=widgets.Layout(width="140px", height="36px"),
    )
    out_panel = widgets.Output()
    _active_players = []

    def on_click(_):
        with out_panel:
            clear_output(wait=True)
            text = text_input.value.strip()
            if not text:
                print("请输入文本")
                return
            if not chk_zero.value and not chk_xvec.value:
                print("请至少勾选一种说话人嵌入")
                return

            for p in _active_players:
                p.stop()
            _active_players.clear()

            tasks = []
            if chk_zero.value:
                tasks.append(("零向量（基线）", None,
                               ms.Tensor(np.zeros((1, 512), dtype=np.float16))))
            if chk_xvec.value:
                for i in range(xvec_count.value):
                    idx = xvec_inputs[i].value
                    emb = ms.Tensor(
                        np.array(_xvec_ds[idx]["xvector"], dtype=np.float32)[None],
                        dtype=ms.float16,
                    )
                    tasks.append((f"数据集嵌入 index={idx}", idx, emb))

            short = f"「{text[:40]}{'...' if len(text) > 40 else ''}」"
            print(f"共 {len(tasks)} 路，开始合成 {short}\n")

            for label, idx, emb in tasks:
                fname = "demo_zero.wav" if idx is None else f"demo_xvec_{idx}.wav"
                try:
                    wav, sr = tta_infer(
                        text=text,
                        speaker_embeddings=emb,
                        threshold=thr_slider.value,
                        save_path=os.path.join(OUTPUT_DIR, fname),
                        verbose=False,
                    )
                    logger.info("[%s] 时长: %.2fs", label, len(wav) / sr)
                    player = BoardAudioPlayer(wav, sr, title=label).show()
                    _active_players.append(player)
                except Exception as e:
                    logger.error("合成失败 [%s]: %s", label, e)
                    print(f"合成失败 [{label}]: {e}")
            logger.info("全部合成完毕，共 %d 路", len(_active_players))

    run_btn.on_click(on_click)

    # 布局 
    ui = widgets.VBox([
        _section("文本输入"),
        text_input,
        _section("生成参数"),
        widgets.HBox([thr_slider, thr_desc], layout=widgets.Layout(align_items="center")),
        _section("说话人嵌入"),
        widgets.HBox(
            [chk_zero, chk_xvec, xvec_count],
            layout=widgets.Layout(align_items="center", gap="6px", margin="0 0 6px"),
        ),
        xvec_inputs_box,
        widgets.HTML("<hr style='margin:10px 0;border-color:#ddd'>"),
        run_btn,
        _section("合成结果"),
        out_panel,
    ], layout=widgets.Layout(padding="8px 12px"))

    display(ui)
    logger.info("交互 Demo 已加载")

except ImportError:
    logger.warning("ipywidgets 不可用，执行静态 fallback")
    demo_text = "Welcome to the speech synthesis demo running on Ascend NPU hardware."
    wav, sr = tta_infer(demo_text, save_path=os.path.join(OUTPUT_DIR, "demo_output.wav"))
    BoardAudioPlayer(wav, sr, title=demo_text).show()

HTML(value='\n    <style>\n    .widget-slider .noUi-handle {\n        background: #ff7043 !important; border-c…

2026-04-27 00:28:08,609 [INFO] 交互 Demo 已加载
